# Train.py Testing
Test the training loop with BPR loss on the Two-Tower model.
Run a short training session (5 epochs) to verify:
- Dataset loading
- BPR loss computation
- Model forward pass
- Checkpoint saving

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import torch
from pathlib import Path

from train import train, bpr_loss
from model import TwoTowerModel
from data.dataset import TwoTowerDataset

In [ ]:
# Check that all required data files exist
train_csv = "../data/processed/train.csv"
test_csv = "../data/processed/test.csv"
metadata_json = "../data/processed/metadata.json"

print(f"train.csv exists: {os.path.exists(train_csv)}")
print(f"test.csv exists: {os.path.exists(test_csv)}")
print(f"metadata.json exists: {os.path.exists(metadata_json)}")

# Load and display metadata
with open(metadata_json, "r") as f:
    metadata = json.load(f)

print("\nDataset Statistics:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Test BPR loss computation
print("Testing BPR loss function...\n")

# Create dummy scores
pos_scores = torch.tensor([2.0, 1.5, 0.5])
neg_scores = torch.tensor([0.5, 0.2, -0.5])

loss = bpr_loss(pos_scores, neg_scores)
print(f"pos_scores: {pos_scores}")
print(f"neg_scores: {neg_scores}")
print(f"BPR Loss: {loss.item():.6f}")
print("\n✓ BPR loss computed successfully")

In [ ]:
# Test model forward pass
print("Testing model forward pass...\n")

num_users = metadata["num_users"]
num_movies = metadata["num_movies"]

model = TwoTowerModel(
    num_users=num_users,
    num_movies=num_movies,
    embed_dim=64,
    hidden_dim=64,
    dropout=0.1
)

# Create dummy batch
batch_size = 16
user_ids = torch.randint(0, num_users, (batch_size,))
pos_ids = torch.randint(0, num_movies, (batch_size,))
neg_ids = torch.randint(0, num_movies, (batch_size,))

pos_scores, neg_scores = model(user_ids, pos_ids, neg_ids)

print(f"Batch size: {batch_size}")
print(f"pos_scores shape: {pos_scores.shape}")
print(f"neg_scores shape: {neg_scores.shape}")
print(f"pos_scores sample: {pos_scores[:5]}")
print(f"neg_scores sample: {neg_scores[:5]}")
print("\n✓ Model forward pass successful")

In [ ]:
# Run a short training session (5 epochs)
print("Running short training session (5 epochs)...\n")

# Set output directory for test checkpoints
test_checkpoint_dir = "../checkpoints/test"

model = train(
    train_csv_path="../data/processed/train.csv",
    metadata_json_path="../data/processed/metadata.json",
    output_dir=test_checkpoint_dir,
    num_epochs=5,  # Short test run
    batch_size=128,
    embed_dim=64,
    hidden_dim=64,
    dropout=0.1,
    learning_rate=1e-3,
)

In [ ]:
# Verify that checkpoints and logs were saved
print("\nVerifying saved artifacts...\n")

test_checkpoint_dir = "../checkpoints/test"

model_path = os.path.join(test_checkpoint_dir, "model_final.pt")
history_path = os.path.join(test_checkpoint_dir, "training_history.json")

print(f"model_final.pt exists: {os.path.exists(model_path)}")
print(f"training_history.json exists: {os.path.exists(history_path)}")

if os.path.exists(model_path):
    file_size_mb = os.path.getsize(model_path) / (1024 ** 2)
    print(f"model_final.pt size: {file_size_mb:.2f} MB")

if os.path.exists(history_path):
    with open(history_path, "r") as f:
        history = json.load(f)
    print(f"\nTraining history:")
    print(f"  Epochs: {history['num_epochs']}")
    print(f"  Batch size: {history['batch_size']}")
    print(f"  Learning rate: {history['learning_rate']}")
    print(f"  Losses: {[f'{l:.6f}' for l in history['losses']]}")
    print(f"\n✓ All artifacts saved successfully")

In [ ]:
# Test loading the saved checkpoint
print("Testing checkpoint loading...\n")

# Create fresh model
model_fresh = TwoTowerModel(
    num_users=num_users,
    num_movies=num_movies,
    embed_dim=64,
    hidden_dim=64,
    dropout=0.1
)

# Load checkpoint
model_fresh.load_state_dict(torch.load(model_path, weights_only=True))
print(f"✓ Successfully loaded checkpoint from {model_path}")

# Test inference
model_fresh.eval()
with torch.no_grad():
    test_user_ids = torch.randint(0, num_users, (32,))
    test_pos_ids = torch.randint(0, num_movies, (32,))
    test_neg_ids = torch.randint(0, num_movies, (32,))
    
    pos_scores, neg_scores = model_fresh(test_user_ids, test_pos_ids, test_neg_ids)
    print(f"✓ Inference successful on loaded model")
    print(f"  pos_scores shape: {pos_scores.shape}")
    print(f"  neg_scores shape: {neg_scores.shape}")

## Summary

✓ BPR loss function working correctly
✓ Model forward pass producing correct shapes
✓ Training loop completed successfully
✓ Checkpoints and training history saved
✓ Checkpoint loading and inference working

**Next Steps:**
- Run full training with appropriate hyperparameters on the complete dataset
- Monitor loss curves for convergence
- Proceed to evaluate.py to compute Recall@K, Hit Rate@K, MRR@K metrics